In [1]:

!pip -q install pypdf sentence-transformers google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 1.8 MB/s eta 0:00:00


In [3]:

from google.colab import userdata
from google import genai

client = genai.Client(
    api_key=userdata.get("GEMINI_APIKEY")
)

print("Gemini API client is ready.")


Gemini API client is ready.


In [4]:

from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_path = next(iter(uploaded))

reader = PdfReader(pdf_path)
text = "\n".join(page.extract_text() or "" for page in reader.pages)

print("Total pages:", len(reader.pages))
print("Total extracted characters:", len(text))
print("\nFirst 500 characters:\n", text[:500])


Saving sample_student_services_policy.pdf to sample_student_services_policy.pdf
Total pages: 5
Total extracted characters: 4811

First 500 characters:
 Student Services and Campus Facilities Policy
Fictional sample document for educational use
Sample policy document for a fictional institution. Institution: Greenfield Institute of Applied Learning.
Policy version: 2.1. Effective date: 1 July 2026. Review date: 30 June 2027.
This document describes selected student-service procedures and campus rules. It is a fictional training
document created for a retrieval-augmented generation (RAG) assignment. The facts in this document
apply only to this s


In [5]:

def make_chunks(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

chunks = make_chunks(text, chunk_size=800, overlap=150)

print("Chunk size: 800")
print("Overlap: 150")
print("Number of chunks:", len(chunks))
print("\nFULL EXAMPLE CHUNK:\n")
print(chunks[0])


Chunk size: 800
Overlap: 150
Number of chunks: 8

FULL EXAMPLE CHUNK:

Student Services and Campus Facilities Policy
Fictional sample document for educational use
Sample policy document for a fictional institution. Institution: Greenfield Institute of Applied Learning.
Policy version: 2.1. Effective date: 1 July 2026. Review date: 30 June 2027.
This document describes selected student-service procedures and campus rules. It is a fictional training
document created for a retrieval-augmented generation (RAG) assignment. The facts in this document
apply only to this sample institution.
The Student Services Office is located in Block B, Room 104. Office hours are Monday to Friday, 9:00
a.m. to 4:30 p.m., excluding declared campus holidays. Students must show their campus ID card when
requesting a service in person.
The official email for student service requests


In [6]:

from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Number of chunk texts:", len(chunks))
print("Embedding matrix shape:", chunk_embeddings.shape)
print("Embeddings and their corresponding chunk texts are ready.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of chunk texts: 8
Embedding matrix shape: (8, 384)
Embeddings and their corresponding chunk texts are ready.


In [7]:

def retrieve(question, chunks, embeddings, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    scores = embeddings @ question_embedding
    indices = np.argsort(scores)[::-1][:top_k]

    return [(int(i), float(scores[i]), chunks[i]) for i in indices]

question = "How much does a replacement campus ID card cost?"

results = retrieve(question, chunks, chunk_embeddings)

for rank, (idx, score, chunk) in enumerate(results, start=1):
    print(f"\nRank {rank} | Chunk {idx} | Similarity: {score:.4f}")
    print(chunk)
    print("-" * 60)



Rank 1 | Chunk 0 | Similarity: 0.4829
Student Services and Campus Facilities Policy
Fictional sample document for educational use
Sample policy document for a fictional institution. Institution: Greenfield Institute of Applied Learning.
Policy version: 2.1. Effective date: 1 July 2026. Review date: 30 June 2027.
This document describes selected student-service procedures and campus rules. It is a fictional training
document created for a retrieval-augmented generation (RAG) assignment. The facts in this document
apply only to this sample institution.
The Student Services Office is located in Block B, Room 104. Office hours are Monday to Friday, 9:00
a.m. to 4:30 p.m., excluding declared campus holidays. Students must show their campus ID card when
requesting a service in person.
The official email for student service requests
------------------------------------------------------------

Rank 2 | Chunk 5 | Similarity: 0.4352
umbing, electrical, or furniture problems to
the hostel offic

In [17]:
def answer_question(question, chunks, embeddings, top_k=3):
    results = retrieve(question, chunks, embeddings, top_k)

    context = "\n\n".join(
        f"[Chunk {idx}]\n{chunk}"
        for idx, score, chunk in results
    )

    prompt = f"""
You are a question-answering assistant for one PDF document.

Answer using ONLY the supplied context.
Do not use outside knowledge or make assumptions.
If the context does not explicitly contain enough information
to answer the question, reply exactly:
"I cannot find the answer in the provided document."

Cite the supporting chunk as [Chunk N] when answering.
If the answer is unavailable, do not invent a citation.

CONTEXT:
{context}

QUESTION:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    print("QUESTION:", question)
    print("\nANSWER:\n", response.text)
    print("\nRETRIEVED CHUNKS AND SCORES:")

    for idx, score, chunk in results:
        print(f"\nChunk {idx} | Similarity: {score:.4f}")
        print(chunk[:250].replace("\n", " "), "...")

In [20]:
import time
from google.genai.errors import ServerError

def answer_question_with_retry(q, chunks, chunk_embeddings, retries=3, delay=2):
    for attempt in range(retries):
        try:
            answer_question(q, chunks, chunk_embeddings)
            return
        except ServerError as e:
            if e.code == 503 and attempt < retries - 1:
                print(f"\n[Model busy, retrying in {delay} seconds... (Attempt {attempt+1}/{retries})]")
                time.sleep(delay)
                delay *= 2
            else:
                raise e

in_document_questions = [
    "How much does a replacement campus ID card cost?",
    "How many books can an undergraduate borrow, and for how many days?",
    "How long does a printed transcript request take to process?"
]

out_document_questions = [
    "What is the semester tuition fee?",
    "How many sick days may a hostel resident take each year?",
    "What is the penalty for arriving late to a computer laboratory?"
]

print("=" * 70)
print("IN-DOCUMENT QUESTIONS")
print("=" * 70)

for q in in_document_questions:
    answer_question_with_retry(q, chunks, chunk_embeddings)
    print("\n" + "=" * 70)

print("\nOUT-OF-DOCUMENT QUESTIONS")
print("=" * 70)

for q in out_document_questions:
    answer_question_with_retry(q, chunks, chunk_embeddings)
    print("\n" + "=" * 70)

IN-DOCUMENT QUESTIONS

[Model busy, retrying in 2 seconds... (Attempt 1/3)]
QUESTION: How much does a replacement campus ID card cost?

ANSWER:
 A replacement campus ID card costs ■150 [Chunk 5].

RETRIEVED CHUNKS AND SCORES:

Chunk 0 | Similarity: 0.4829
Student Services and Campus Facilities Policy Fictional sample document for educational use Sample policy document for a fictional institution. Institution: Greenfield Institute of Applied Learning. Policy version: 2.1. Effective date: 1 July 2026. R ...

Chunk 5 | Similarity: 0.4352
umbing, electrical, or furniture problems to the hostel office. Urgent electrical faults must also be reported immediately to the duty warden. Students must not attempt electrical repairs themselves. The hostel maintenance desk operates Monday to Sat ...

Chunk 6 | Similarity: 0.4238
ys. A printed transcript request costs ■100 for the first copy and ■40 for each additional copy ordered in the same request. Transcript requests are submitted to the Student 

The chatbot answered the questions whose information was available in the PDF and should have refused the questions whose answers were not present. I checked whether the retrieved chunks contained the relevant facts. The similarity scores helped me understand how closely the chunks matched each question.

When I increased the chunk size from 800 to 3000 characters, the number of chunks decreased. The similarity scores changed because each embedding represented a larger section of text. Larger chunks sometimes provide more context, but they may also contain unrelated information. I compared the answers and retrieved chunks to decide which chunk size worked better for my document.